## ATracker Tutorial

ATracker works with folders of videos, or better said, projects. It was designed to handle large folders of videos to be tracked with many different parameters that may furthermore differ between videos, such as having refuges or not, the number of objects to track etc.

This Jupyter notebook provides a **step-by-step tutorial** on using ATracker, covering everything from loading videos to tracking and processing data. We will work with **six example video files** designed specifically for this tutorial. These files will guide you through:
- Drawing the **region of interest (ROI)**
- Creating and adjusting **masks** for specific areas (e.g., refuge zones)
- Setting up **zones** for tracking structures in the arena
- Defining **walls** for circular arenas
- Configuring different **thresholding methods** for Background subtraction tracking, Color-based tracking, and  Barcode tracking (to implement).
- Running **drymode** for testing tracking parameters
- Running tracking
- And manual tracking mode

### 1. Setting Up

As Atracker is still under active development we will load it locally.

> Note that when running atracker (and dependencies) for the first time it make take up to 30s to load everything after `import atracker`, but this is just for the first load

We will use a **universal method** to locate the ATracker folder **without manually specifying system-dependent paths**. This ensures the tutorial works seamlessly across different machines and operating systems.

In [ ]:
import sys
import os

# Get the user's home directory dynamically
home_dir = os.path.expanduser("~")

# Define the ATracker path inside the home directory
atracker_folder = os.path.join(home_dir, "atracker")

# Append ATracker to the system path
sys.path.append(atracker_folder)

# Import ATracker
import atracker

If might be you have the atracker folder in a Google drive. To automatically load that version, we can use the following code instead:

In [ ]:
import os
import sys
from pythutils.sysutils import get_google_drive_path

# Load atracker
drive_path = get_google_drive_path()
atracker_folder = os.path.join(drive_path, "ABEClab", "atracker")
sys.path.append(atracker_folder)
import atracker

Now we can initialize the **video folder**, which will contain the files to be processed using ATracker. For this tutorial, to ensure that we work with a clean dataset without modifying the original tutorial files:

1. **Copy the example video files** (only the `.mp4` files) from the `0originals/` folder.
2. **Paste them into a new folder** called `"test"` on your **Desktop**.

Then run:

In [ ]:
import os

# Get the user's home directory
home_dir = os.path.expanduser("~")

# Define the video folder path (inside "test" on the Desktop)
video_folder = os.path.join(home_dir, "Desktop", "test")

# Load ATracker with the video folder
AT = atracker.ATracker(video_folder)

ATracker will have initiated the folder for tracking, including creating a number of files, about which it prints some statements in the output window. The folders it creates are as follows:

- **0originals**: stores all raw files used for tracking. This includes the raw .h264 and converted .mp4 videos but also for example background files or mask files. mp4 files are automatically moved to this folder, ready for tracking.
- **1todo**: for (temporally) storage of videos that need to be tracked. You can for example decide to track a couple specific videos, the you can simply drag them there and in python tell tracking to look at the todo folder (explained below).
- **2temp**: is a folder only used by atracker. It moves relevant files it needs to tracking to this folder during tracking, and after finishing it moves the files again.
- **3tracked**: is the folder that contains all csv tracking files as well as the output videos from tracking.
- **4processed**: contains all the csv files created from the tracked csv files using the processing function.

Besides these folders, you will see a couple new files:

- **_config.conf**: simple text file that contains all relevant parameters used for tracking. These configuration settings can be manually changed here or using the `set_config` function.
- **_overview.xslx**: a template excel file to fill with relevant data about all the videos. All the videos we want to track should be in this table. We can easily add this list as well as extract all relevant information using the `setup_files` function (explained below)
- **_treshinfo.yml**: for storing all relevant information regarding the objects to track and is mainly used by atracker automatically.

### 2. Setting up the video files

The next thing to do is to load the video files. MP4 videos are already automatically moved to the "0 originals" folder, but they are not yet in the overview file. To add them while also extracting relevant information, such as on video width, height, fps, and duration we use the `setup_files` function.

This function takes a number of parameters to help extract the right variable names from the filename and potentially convert the videos as well:
-  `fname_extract`: if the variables should be extracted from the filename (True/False)
-  `fname_vars`: list of variables to be extracted (a tuple of strings)
-  `fname_sep`: the charater that separates the components of the filename (a string)
-  `autoconvert`: if videos should be automatically converted to .mp4 (True/False)
-  `skip`: if videos already in the overview file should be skipped from loading (True/False)

For example, if I have files with a specific filename structure such as `tempexp_250523_setup1_143024_S01.h264`, I can automatically extract the variables and add them to the overview file as follows:

In [ ]:
AT.setup_files(fname_vars = ("exp","name","date","session","time"), fname_sep = "_")

If videos are recorded with Raspberry Pi, such as using the **pirecorder** package, then these will be .h264 format. These can thus be automatically converted to .mp4 using the `autoconvert` parameter:

In [ ]:
AT.setup_files(fname_extract=False, autoconvert = True)

You can now check the overview excel file to indeed see the five videos are added as well as relevant video information and the variables are added to the respective columns. You can also access it directly in Python via the following command:

In [ ]:
AT.overview

In cases where we update the excel file directly while being in an active Python session, we need to make sure it is updating the loaded overview. To do this use the `reload` function:


In [ ]:
# After having made changes to the excel file directly, load the updated overview:
AT.reload()

To show the overview information for a single video, the `showinfo` function can be used:

In [ ]:
AT.showinfo("animtest_regions_071024_S02_1205")

### 3. Listing files and getting file indices

To get a list of videos with their indices, the `get_files` function can be used. A couple special parameters can be used to select only specific videos. This helps enormously when needing to only update the roi on different days or the mask based on the setup, or to set different treshold parameters for files in the morning versus afternoon. For this we can use the `inds`, `query`, and `cats` parametes as explained below.

To simply show all files with full path and their indices you can use the get_files function:

In [ ]:
AT.get_files()

We can also get the full paths of the files as if they were in a different folder, such as in the temp directory, using the cdir parameter. For example:

In [ ]:
AT.get_files(cdir="temp")

In a similar way we can change the filetype in the generated list of names, for example to change .mp4 to .csv:

In [ ]:
AT.get_files(filetype=".csv")

These above functions are particularly handy in the context to check if files exist. For this you can use the `exists_only` parameter and set it to `True`. For example, to check what files of the overview exist in the tracked folder:

In [ ]:
AT.get_files(cdir="tracked", filetype=".csv", existonly=True)

We can also simply show a subset of files based on their indices:

In [ ]:
AT.get_files(inds=[2,3,4])

If we want the opposite, i.e. get the indices of files, we can use the get_inds function:

In [ ]:
AT.get_inds("animtest_regions_071024_S02_1205")

Another way to select files is to use the `query` parameter and use column names and criteria. For example, we can use the skip column to only select videos that should not be skipped"

In [ ]:
AT.get_files(query="skip==0")

or use the "time" column to only select videos taken before a certain time (make sure the column is converted first):

In [ ]:
AT.overview['time'] = AT.overview['time'].astype(int)
AT.get_files(query="time <= 1300")

Finally we can subset based on categories. This is very handy as it will automatically select the first video of all videos that are unique in terms of that category. For example, if there are 6 videos of two trials, it will show the video information of two videos, the first of each trial. This is helpful if video settings remain the same within the category and thus for example the roi only needs to be set for each session:

In [ ]:
AT.get_files(cats=["session"])

### 4. ATracker configuration

#### 4.1 Set the configuration file

We can set a lot of parameters for all the different elements to do with tracking. These are all stored dynamically in the python session, but also in the `_config.conf` file, so they can be easily manually updated without having to open python.

To configure the tracking and other settings of atracker in Python the `set_config` function can be used. The documentation explains nicely everything that can be set and in what way:

In [ ]:
print(AT.set_config.__doc__)

We can also just show the current configuration settings:

In [ ]:
print(AT.config)

When setting up your project, it may be handy to set a number of settings, but we can do this also step by step as we go through the different parts of the tracking process. Here is an example of how to set the configuration:

In [ ]:
AT.set_config(fps=24, real_dims=(300, 300), startframe=50, orient_get=False, 
              show_tracking=False, userwait=False, frame_disstep=100, overwrite=False)

#### 4.2 Set number of objects

By default, the number of objects that will be tracked is 1. If we want to change this, we can set the `cusobjects` parameter in the `track` function directly when tracking. In case videos differ in the number of objects that need to be tracked (such as some videos have one object, others have 2 etc) it is best to set that directly in the *objects* column of the overview file. 

We can set the objects directly in the excel file and then reload it (see above) or by using the dedicated `set_objects` function:

In [ ]:
AT.set_objects(inds=[0,3,4], objects=[1,2,3])

#### 4.3 Extract background files

ATracker relies on background substraction for many functionalities. Although some types of tracking do not need it, it is helpful in most cases to have background images for all videos where the animal is not visible. We can do this with the `get_bgfiles` function.

In [ ]:
AT.get_bgfiles()

In most cases the default parameters are fine, namely it will randomly select 25 frames in the whole video to remove any moving objects and thus only keep the real background. But in some cases we may want to use less or more images to get a more reliable background image. This should be set in the configuration under the `bg_frames` parameter. For example, to use 10 images only, use:

In [ ]:
AT.set_config(bg_frames=10)
AT.get_bgfiles()

We can also set the start and stop frames to use for background substraction. For example, sometimes there might be a lot of movement at the start and end of the video that we want to ignore. We can set this with the `starts` and `stops` parameters. For example, to get bgfiles for all videos but within frame 500 to 5000:

In [ ]:
AT.get_bgfiles(starts=[500], stops=[5000])

The newly created background files are automatically stored in the `0originals` folder as well as listed in bg`bgimg` column of the overview file.

#### 4.4 Setting up videos that have multiple separate tracking regions

With ATracker it is possible to track the same video multiple times to track different regions within the video. But since all the tracking file information will be the same (except the roi), some extra information needs to be provided to ATracker that it knows it should check for regions. We do this using the `set_config` function by setting the `regions` parameter to True:

In [ ]:
AT.set_config(regions=True)

Now a new column with `region` is added to the overview file. If regions is set to `False`, the column, if it exists, will be deleted again such that ATracker will not try to look for region information later.

The next step is to add the region information to the overview file. For this we use the `set_regions` function, which takes the parameters `inds` and `nr`, which is a list of indices of the file(s) in the overview to create the regions for and the number of regions to create. For example:

In [ ]:
AT.set_regions(inds=[0,3], nr=4)

and let's look at the change in the overview file

In [ ]:
AT.overview.loc[:,["video","region"]]

Now ATracker is set up to track each video multiple times corresponding to the different regions, and tracked videos with different regions  automatically get the region number appended to the file name e.g. `vidx243_R1.mp4`, `vidx243_R2.mp4`. 

Note that if the number of regions differs between videos, you can either call the `set_regions` function in a loop or add the information manually in the overview file and reload it (`AT.reload()`).


### 5 Setting-up key tracking parameters interactively

A main functionality of ATracker is that it makes it very easy to dynamically set critical tracking parameters for all videos, even if they vary among them. For this there is the `set_interactive` function. This function will open an interactive interface where the video and additional windows and sliders are shown. The mouse can be used interactively, such as to control the sliders or draw on the video. Also many hotkeys are programmed to enable a lot of cuntionality.

In [ ]:
AT.set_interactive()

To show/hide a list of all the options of the interactive video mode as an overlay on the video, simply press the `h` key. Some of the options are to draw lines, rectangles, polygons, circles, ellipses, create a mask file of what is drawn, invert the mask, to visualise the mask or hide it, show a mouse crosshair, show a diagonal cross, and show horizontal and vertical helperlines of where the mouse exactly is. An overview of all possibilities can be called by typing in:

In [ ]:
print(atracker.ivideo.keys.__doc__)

We can draw various shapes, including a line (press `l`), a circle (press `c`), an ellipse (press `v`), a rectangle (press `k`), and a polygon (`p`). All work in the same way by first clicking where you want the shape to start and lift the mouse again where you want it to end.

Note that a great functionality is that we can set specific parameters for specific videos using the `inds`, `query`, and `cats` parameters, as explained above. For example, we may need to create a specific mask for each setup we use, due to small differences in camera position. Or maybe we run two sessions per day and each session has a different position of plants. In those cases we can nicely automatically select the minimal number of videos to set the settings and create the files we need.

Using the `set_interactive` function we can set a variety of parameters, including the framelimits, the roi, the mask etc. We go through each of them and all options in detail below. Note that only a single mode can be set as `True`, such that the interface options are automatically tweaked to display only that what is required

#### 5.1 Set pixel to mm conversion

We can use the `real_dims` configuration parameter to provide the real dimensions in mm of the width and height of the region of interest, which is then automatically used to set the conversion parameter.

Another way is to draw a line of known distance in the video and provide its dimensions using the interactive functionality with the `conv` parameter set to `True`. Here we can set the `conv_mm` parameter to indicate the length of the line(s) as a tuple of values in mm. The interactive drawing mode will open and for each line you draw, click `s` afterwards to link it to the provided conv_mm values.

In [ ]:
AT.set_interactive(conv=True, conv_mm=(300,300))

#### 5.2 Framelimits

Automatically the first and last frames of a video are used as the framelimits to use for tracking. However, in many cases we may want to have more custom framelimmits. Although for this we can also use the `startframe` and `stopframe` parameters of the `set_config` function, in some cases we may want specific limits for specific videos. 

To do this, we set the `framelimits` parameter to `True`. Now a video is shown and with the bracket keys (`[` and `]`) we can set the start and stop frame respectively:

In [ ]:
AT.set_interactive(framelimits=True)

#### 5.3 Region of interest

To set the region of interest, the boundaries of what we want to track, we set the `roi` parameter to True. Then the interactive mode awaits a rectangle to be drawn around the region of interest. When happy click `s` to store it.


In [ ]:
AT.set_interactive(roi=True)

#### 5.4 Mask

To create a mask we can draw with the mouse in various ways. We can use the variety of shapes available. We can also add multiple shapes sequentially to the layer by each time after drawing entering `a` to add. To show the mask type in `m`. Finally, it is possible to invert the mask by clicking the `i` key. The mask will be stored in the originals folder as a black and white image.

In [ ]:
# Draw a mask for the area of the refuge, which will be excluded for tracking 
AT.set_interactive(mask=True, inds=[7])

Note that when adding shapes to the mask, the black area will be the mask and excluded, so when wanting to exclude everything outside of an area you need to invert the mastk (simply press `i`).

#### 5.5 Walls 

In a similar way we can draw the exact location of walls. This can be helpful when the tank is not rectangular or positioned horizontally, or when there are additional walls that we want to get information about. 

In [ ]:
# Draw a circular walls mask to get information regarding distance to the perimeter of the circular arena
AT.set_interactive(walls=True, inds=[1])

#### 5.6 Zones 

We can draw zones if we are interested in tracking objects in (relation to) multiple areas in the video.

In general the mask parameter can be set to create a mask, such as to cover a plant refuge. If you want to have a second type of mask that is different from what is masked, such as to distinguish between refuges and structures, then we can simply use the `maskzone` parameter:

In [ ]:
# Use the mask functionality to set the stones as one zone
AT.set_interactive(maskzone=True, inds=[7])

If we want to add multiple zones, with each consisting of one area or object, we can use the `zones` parameter. This still works the same way as creating a mask, but now rather than adding each object to the mask, we click `o` after drawing each area/object to temporarily store the coordinates of the mask of that zone. 

So draw with the desired drawing tool your first zone, click the `m` key to show the mask, click the `a` key to add the drawing to the mask, click the `o` key to extract the coordinates from the mask, and continue drawing the next zones. When finished, click `s` and a single mask-type image will be created, but with the zones displayed as different colors.

In [ ]:
# Use the mask functionality to set zones for each individual stone
AT.set_interactive(zones=True, inds=[7])

Note that, if zones overlap, more recently drawn zones have priority and the underlying zones will lack the overlapping region.

#### 5.7 Points of interest

We can finally use the interactive drawing function to draw specific points of interest, such as for calculating distance to specific points. For this we use the `getpt` parameter. It will store as many columns in the overview file as points are clicked before `s` is entered. You can also already set the names of the columns with the `ptcolnames` parameter, which should be a list of strings the same length as the numer of points of interest.

In [ ]:
AT.set_interactive(getpts=True, ptcolnames=["point1","point2"])

### 6. Setting-up tracking treshold parameters

Finally, it is time to set the `treshtypes` parameter. The default is tresholding based on background substraction, which looks at how different pixels are in black and white (`bw`). But it is also possible to track in colors, thereby simply provide the color that needs to be tracked, and I am implementing the possibility for barcode tracking.

Besides the video window, it will show a frame position slider, which adjusts the position of the video, a tresholding panel, to help create optimal parameters to track the object of interest, a small tresholded image, which is how the image looks like after applying the treshold settings, and a contour info window with information about the potentially identified objects.

Key is to get the same number of contours as objects that we want to track, and that this is as stable as possible across the video, thus including when it is darker or lighter due to different lighting in parts of the video. Also make sure the size parameters are correct such that potentially much smaller and larger objects will be ignored. In general it is good to blur the image (e.g. set to `5`) as this eliminates noise and smooths the shape, and then erode it (e.g. set to `8`) which reduces the size of all contours, which were increased due to blurring. In some cases a second round of blurring is necessary to get rid of additional noise. Finally, set the treshold parameter. A higher treshold is more critical what will be included or not.

Contours that would be tracked with current settings are shown in blue and with area information in the contour info box, while contours that would be excluded are shown in red. Note that the mask will be taken into account already during tresholding, so if you can see the object and you are sure the settings should identify it, then maybe it is behind the mask. When happy with the parameters, click `s` and all trashold values are stored in the special `treshinfo.yml` file.

Note that in most cases under laboratory conditions you only need to set the treshold parameters once, although sometimes when videos are not optimal it may be necessarily to track certain videos with slightly adjusted treshold parameters. After saving you can run the interactive function again on different video instances and scroll through the video using the interface to double check it is working for those as well.

In [ ]:
# Set treshold parameters for black and white tracking
AT.set_interactive(treshtypes=["bw"], inds=[7])

In [ ]:
# Set multiple color treshold parameters in series
AT.set_interactive(treshtypes=["red","blue","green","orange","black","brown"], inds=[1])

In case you feel the need to set different parameters for standard tresholding, simply change the treshtype to start with "bw_" and add any term you want. For example "bw_lightissue" for the videos that have an issue with being brighter than others. As the tresh_type starts with "bw", it is automatically initiated as standard tresholding.

In [ ]:
# Set treshold parameters for additional black and white tracking
AT.set_interactive(treshtypes=["bw_strict"], inds=[7])

In [ ]:
AT.overview.loc[:,["video","tresh_types"]]

To link the video to a specific treshtype, the `tresh_types` column of the AT.overview should be set. By default empty cells are interpreted as `bw` treshtype, which is what is used in most cases. But if you want to track a video with objects in specific colors or barcodes state this in the tresh_types column. The values should be a list of strings, e.g. ["red", "green,black", "blue", "bw"]. In python you can simply change the tresh_types column of the overview file directly like this:

In [ ]:
AT.overview.tresh_types = ["bar", "(red,green,blue)"] + ["bw"] * 6
AT.save()

You can also update the overview file manually and run the `AT.reload()` function to update it python.

### Run processing

Run manual tracking to fix erroneous tracking data if visual inspection of videos indicated this is the case. You can thereby provide a specific list of files if needed.

In [ ]:
# Provide an overview of parameters for the processing function
print(AT.process.__doc__)

In [ ]:
from pythutils.fileutils import listfiles 
listfiles(AT.dirs["tracked"], type=".csv", keepdir=True)

In [ ]:
AT.process(names = ["animtest_solo_071024_S01_1413.csv"], manfix=True, manonly=True, man_types = ["c"], overwrite=True)

In [ ]:
# Add dummy tracking data to the white video
AT.process(names = ["white_video.csv"], manfix=True, manonly=True, man_types = ["c"], overwrite=True)